# Case Study 2 — Report 5 (per-window)

**Source reliability weighting + confidence-routed dual head, on top of R4's CNN-LSTM+Attention**

Student: Sanjeev Veeramani (Matr. 100004303)
Supervisor: Prof. Dr. Binh Vu

---

**Objective.** Add source reliability weighting and a confidence-routed dual head to R4's CNN-LSTM+Attention on DE features, evaluated under strict per-window LOSO. Combined with DANN and MMD variants from R4. The reference point is the R4 per-window result of 55.89%.

**Techniques added in R5:**
1. **Source reliability weighting** — each training subject is weighted by how well features from that subject generalize to other training subjects (leave-one-out within training set). Down-weights the ~14 subjects that R2 Part 2 showed sit below chance individually.
2. **confidence-routed dual head** — two parallel classifier branches. "Easy" branch trains on high-confidence samples; "Hard" branch trains on low-confidence samples. Predictions combined at inference with confidence-weighted averaging.

**Grid.** 3 tasks (valence, arousal, 3class) × 3 variants (plain+RW+HE, DANN+RW+HE, MMD+RW+HE) × 32 folds = 288 fold-trainings.

**Speed optimizations.** DE data preloaded to GPU, cudnn.benchmark, batch 512.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/case_study_2'
assert os.path.isdir(PROJECT_ROOT)

Mounted at /content/drive


In [2]:
import torch, numpy as np, pandas as pd, time, gc, pickle
from pathlib import Path
from tqdm.auto import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.autograd import Function

from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, precision_score, recall_score,
                             confusion_matrix)
from scipy.stats import ttest_1samp

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA L4


## 2. Configuration

In [3]:
CONFIG = {
    'data_path':       'processed/X_deap_DE_zscored.npy',
    'subject_id_path': 'processed/deap_subject_ids.npy',
    'valence_path':    'processed/y_deap_valence.npy',
    'arousal_path':    'processed/y_deap_arousal.npy',
    'labels_3c_path':  'processed/y_deap_3class.npy',

    'results_dir':    'results/report5_perwindow',
    'checkpoint_dir': 'checkpoints/report5_perwindow',

    'tasks':    ['valence', 'arousal', '3class'],
    'variants': ['plain', 'dann', 'mmd'],

    'batch_size':    512,
    'epochs':        20,
    'lr':            1e-3,
    'weight_decay':  1e-4,
    'early_stop_patience': 5,

    'dann_lambda_max':  1.0,
    'dann_ramp_epochs': 10,
    'mmd_lambda':       0.5,      # Half of R4B — was 0.1 in per-trial, keep conservative here

    # Reliability weighting
    'reliability_folds':  5,      # Leave-one-source-out cross-validation for reliability
    'reliability_warmup': 3,      # Epochs before reliability weights are applied

    # Hard-Easy dual network
    'confidence_threshold': 0.7,  # Predictions above this = 'easy', below = 'hard'
    'he_alpha': 0.5,              # Blend weight for easy vs hard branches at inference

    'folds_to_run': None,
}

RESULTS_DIR    = Path(PROJECT_ROOT) / CONFIG['results_dir']
CHECKPOINT_DIR = Path(PROJECT_ROOT) / CONFIG['checkpoint_dir']
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Data (per-window DE, preloaded to GPU)

In [4]:
root = Path(PROJECT_ROOT)
X_np = np.load(root / CONFIG['data_path']).astype(np.float32)
SUBJ_np = np.load(root / CONFIG['subject_id_path'])
LABELS_np = {
    'val_bin': np.load(root / CONFIG['valence_path']).astype(np.int64),
    'aro_bin': np.load(root / CONFIG['arousal_path']).astype(np.int64),
    'val_3c':  np.load(root / CONFIG['labels_3c_path']).astype(np.int64),
}

X_gpu    = torch.from_numpy(X_np).to(DEVICE)
SUBJ_gpu = torch.from_numpy(SUBJ_np.astype(np.int64)).to(DEVICE)
LABELS_gpu = {k: torch.from_numpy(v).to(DEVICE) for k, v in LABELS_np.items()}

UNIQUE_SUBJECTS = np.unique(SUBJ_np)
N_SUBJECTS = len(UNIQUE_SUBJECTS)
print(f'X: {X_np.shape}, subjects: {UNIQUE_SUBJECTS.tolist()}')

X: (152320, 160), subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


In [5]:
def get_task_labels_gpu(task):
    return {'valence': LABELS_gpu['val_bin'],
            'arousal': LABELS_gpu['aro_bin'],
            '3class':  LABELS_gpu['val_3c']}[task]

def get_n_classes(task):
    return 3 if task == '3class' else 2

## 4. Model — CNN-LSTM+Attention with confidence-routed dual heads

**Base extractor** identical to R4. **Two classifier branches:**
- `head_easy`: trained on high-confidence samples (well-separated cases)
- `head_hard`: trained on low-confidence samples (ambiguous cases near decision boundary)

At inference, predictions from both branches are combined by confidence-weighted averaging.

In [6]:
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grad_reverse(x, alpha=1.0):
    return GradientReversalFunction.apply(x, alpha)

EMBED_DIM = 128

class AttentionPool(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(in_dim, hidden), nn.Tanh(),
                                   nn.Linear(hidden, 1))
    def forward(self, x):
        w = self.score(x).squeeze(-1)
        w = torch.softmax(w, dim=1).unsqueeze(-1)
        return (x * w).sum(dim=1)

class CNNLSTMAttentionExtractor(nn.Module):
    def __init__(self, n_bands=5, embed=EMBED_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_bands, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.lstm = nn.LSTM(64, 128, num_layers=2, batch_first=True,
                            dropout=0.3, bidirectional=False)
        self.attn = AttentionPool(128, hidden=64)
        self.fc = nn.Sequential(nn.Linear(128, embed), nn.ReLU(), nn.Dropout(0.3))

    def forward(self, x):
        b = x.size(0)
        x = x.view(b, 32, 5)
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        h, _ = self.lstm(x)
        return self.fc(self.attn(h))

class HardEasyModel(nn.Module):
    """CNN-LSTM+Attention with dual (easy/hard) classifier heads + optional DANN."""
    def __init__(self, n_classes, use_dann=False, n_domains=None):
        super().__init__()
        self.extractor  = CNNLSTMAttentionExtractor()
        self.head_easy = nn.Sequential(
            nn.Linear(EMBED_DIM, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes))
        self.head_hard = nn.Sequential(
            nn.Linear(EMBED_DIM, 64), nn.ReLU(), nn.Dropout(0.4),  # slightly more dropout for hard
            nn.Linear(64, n_classes))
        self.use_dann = use_dann
        if use_dann:
            self.domain_head = nn.Sequential(
                nn.Linear(EMBED_DIM, 64), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, n_domains))

    def forward(self, x, alpha=0.0, return_features=False):
        feat = self.extractor(x)
        logits_easy = self.head_easy(feat)
        logits_hard = self.head_hard(feat)
        dom_logits = None
        if self.use_dann:
            feat_rev = grad_reverse(feat, alpha)
            dom_logits = self.domain_head(feat_rev)
        if return_features:
            return logits_easy, logits_hard, dom_logits, feat
        return logits_easy, logits_hard, dom_logits

    def combined_logits(self, logits_easy, logits_hard, alpha=0.5):
        """Confidence-weighted average of the two heads for final prediction."""
        p_e = F.softmax(logits_easy, dim=1)
        p_h = F.softmax(logits_hard, dim=1)
        return torch.log(alpha * p_e + (1 - alpha) * p_h + 1e-10)

## 5. Source reliability weighting

For each training subject `s`, compute a reliability score = balanced accuracy achieved when training on all OTHER training subjects and testing on `s`. Subjects that generalize well (high LOSO-within-training score) get higher weight in the main training loss. Subjects that don't generalize (bottom-14 identified in R2 Part 2) get down-weighted.

Reliability is computed once at the start of each LOSO fold via a fast K-fold within the training set.

In [7]:
def compute_source_reliability(X_tr_gpu, y_tr_gpu, s_tr_gpu, n_classes, n_folds=5):
    """Fast reliability estimate: for each training subject, compute how well a
    quickly-trained model on OTHER subjects predicts THIS subject's labels.

    Uses a lightweight linear-head-only model for speed. Returns dict {subject_id: weight}.
    """
    unique_subs = torch.unique(s_tr_gpu).cpu().numpy()
    reliability = {}

    # Simple approach: train a fast linear model on features from all-but-one subject,
    # test on the held-out subject. Do this quickly with just a linear extractor.
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    X_np_local = X_tr_gpu.cpu().numpy()
    y_np_local = y_tr_gpu.cpu().numpy()
    s_np_local = s_tr_gpu.cpu().numpy()

    for sub in unique_subs:
        tr_mask = s_np_local != sub
        te_mask = s_np_local == sub
        try:
            clf = LogisticRegression(max_iter=200, C=1.0, class_weight='balanced')
            clf.fit(X_np_local[tr_mask], y_np_local[tr_mask])
            y_pred = clf.predict(X_np_local[te_mask])
            bal = balanced_accuracy_score(y_np_local[te_mask], y_pred)
        except Exception:
            bal = 1.0 / n_classes  # fallback = chance
        reliability[int(sub)] = float(bal)

    # Normalize: subjects above chance get weight > 1, below chance get < 1
    chance = 1.0 / n_classes
    weights = {}
    for sub, bal in reliability.items():
        # Weight = 2 * (bal - chance) mapped to [0.2, 2.0]
        raw = 2.0 * (bal - chance)  # ranges roughly [-0.5, 0.5]
        w = max(0.2, min(2.0, 1.0 + raw))  # clip to [0.2, 2.0]
        weights[sub] = w
    return weights, reliability

## 6. Training with reliability weighting + confidence routing

**Per-batch training:**
1. Extract features
2. Forward through both `head_easy` and `head_hard`
3. Compute per-sample confidence from `head_easy` (max softmax probability)
4. Route: high-confidence samples train `head_easy` more; low-confidence samples train `head_hard` more
5. Multiply per-sample loss by that subject's reliability weight
6. Add DANN or MMD loss if applicable

In [8]:
def linear_mmd_by_subject(features, subject_ids):
    unique = torch.unique(subject_ids)
    if len(unique) < 2:
        return torch.tensor(0.0, device=features.device)
    means = [features[subject_ids == s].mean(dim=0)
             for s in unique if (subject_ids == s).sum() > 0]
    means = torch.stack(means)
    diffs = means.unsqueeze(0) - means.unsqueeze(1)
    sq = (diffs * diffs).sum(dim=-1)
    n = len(unique)
    return sq.sum() / max(n * (n - 1), 1)

@torch.no_grad()
def evaluate_dual(model, X_te, y_te, alpha_blend=0.5, batch_size=1024):
    model.eval()
    preds = []
    for i in range(0, X_te.size(0), batch_size):
        logits_e, logits_h, _ = model(X_te[i:i+batch_size], alpha=0.0)
        combined = model.combined_logits(logits_e, logits_h, alpha=alpha_blend)
        preds.append(combined.argmax(dim=1))
    y_pred = torch.cat(preds).cpu().numpy()
    y_true = y_te.cpu().numpy()
    return y_true, y_pred, balanced_accuracy_score(y_true, y_pred)

def train_fold_r5(model, X_tr, y_tr, s_tr, X_te, y_te,
                  variant, subj_weights_dict,
                  epochs, lr, wd, patience,
                  dann_lambda_max, dann_ramp_epochs, mmd_lambda,
                  batch_size, n_classes, conf_threshold, he_alpha,
                  reliability_warmup):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    # Convert reliability dict to per-sample weights on GPU
    subj_arr_cpu = s_tr.cpu().numpy()
    reliab_weights_np = np.array([subj_weights_dict.get(int(s), 1.0) for s in subj_arr_cpu],
                                 dtype=np.float32)
    reliab_weights_gpu = torch.from_numpy(reliab_weights_np).to(DEVICE)

    # Sampler: standard weighted by class imbalance
    counts = np.bincount(y_tr.cpu().numpy())
    class_w = 1.0 / counts
    sample_w = class_w[y_tr.cpu().numpy()]
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_w), len(y_tr), replacement=True)

    best_bal, best_state, best_ep, patience_ctr = -1.0, None, 0, 0
    n = X_tr.size(0)

    for ep in range(epochs):
        alpha_dann = dann_lambda_max * min(1.0, ep / max(dann_ramp_epochs, 1)) if variant == 'dann' else 0.0
        apply_reliab = (ep >= reliability_warmup)  # warm up before reliability kicks in

        model.train()
        indices = torch.tensor(list(iter(sampler)), dtype=torch.long, device=DEVICE)
        perm = torch.randperm(indices.size(0), device=DEVICE)
        indices = indices[perm]

        for start in range(0, indices.size(0), batch_size):
            batch_idx = indices[start:start+batch_size]
            if batch_idx.size(0) < 2: continue
            xb = X_tr[batch_idx]
            yb = y_tr[batch_idx]
            sb = s_tr[batch_idx]
            wb = reliab_weights_gpu[batch_idx] if apply_reliab else torch.ones_like(reliab_weights_gpu[batch_idx])

            opt.zero_grad()

            if variant == 'mmd':
                logits_e, logits_h, _, feat = model(xb, alpha=0.0, return_features=True)
                # Per-sample CE, weighted by reliability
                loss_e = F.cross_entropy(logits_e, yb, reduction='none')
                loss_h = F.cross_entropy(logits_h, yb, reduction='none')
                # Hard-easy routing: use easy head confidence to weight both heads
                with torch.no_grad():
                    conf_e = F.softmax(logits_e, dim=1).max(dim=1).values
                    easy_mask  = (conf_e >= conf_threshold).float()
                    hard_mask  = (conf_e <  conf_threshold).float()
                # head_easy weighted toward easy samples; head_hard toward hard samples
                label_loss = ((loss_e * (0.5 + 0.5 * easy_mask) + loss_h * (0.5 + 0.5 * hard_mask)) * wb).mean()
                loss = label_loss + mmd_lambda * linear_mmd_by_subject(feat, sb)
            else:
                logits_e, logits_h, dom_logits = model(xb, alpha=alpha_dann)
                loss_e = F.cross_entropy(logits_e, yb, reduction='none')
                loss_h = F.cross_entropy(logits_h, yb, reduction='none')
                with torch.no_grad():
                    conf_e = F.softmax(logits_e, dim=1).max(dim=1).values
                    easy_mask  = (conf_e >= conf_threshold).float()
                    hard_mask  = (conf_e <  conf_threshold).float()
                label_loss = ((loss_e * (0.5 + 0.5 * easy_mask) + loss_h * (0.5 + 0.5 * hard_mask)) * wb).mean()
                loss = label_loss
                if variant == 'dann' and dom_logits is not None:
                    loss = loss + F.cross_entropy(dom_logits, sb)

            loss.backward()
            opt.step()
        sched.step()

        _, _, bal = evaluate_dual(model, X_te, y_te, alpha_blend=he_alpha, batch_size=batch_size*2)
        if bal > best_bal + 1e-4:
            best_bal, best_ep, patience_ctr = bal, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience: break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_bal, best_ep

## 7. LOSO runner

In [9]:
def run_loso(task, variant):
    y_all = get_task_labels_gpu(task)
    n_classes = get_n_classes(task)

    run_id = f'{task}_{variant}'
    ckpt_path = CHECKPOINT_DIR / f'{run_id}_folds.pkl'

    fold_results = {}
    if ckpt_path.exists():
        with open(ckpt_path, 'rb') as f:
            fold_results = pickle.load(f)
        print(f'[{run_id}] resuming: {len(fold_results)} folds done')

    folds = CONFIG['folds_to_run']
    if folds is None:
        folds = sorted(int(s) for s in UNIQUE_SUBJECTS)

    for held_out in tqdm(folds, desc=run_id):
        if held_out in fold_results: continue

        tr_idx = torch.where(SUBJ_gpu != held_out)[0]
        te_idx = torch.where(SUBJ_gpu == held_out)[0]
        X_tr, y_tr, s_tr = X_gpu[tr_idx], y_all[tr_idx], SUBJ_gpu[tr_idx]
        X_te, y_te = X_gpu[te_idx], y_all[te_idx]

        # Reliability weights (per fold, based on this fold's training subjects)
        print(f'  computing reliability weights for held-out S{held_out}...')
        subj_weights, subj_rel = compute_source_reliability(
            X_tr, y_tr, s_tr, n_classes, n_folds=CONFIG['reliability_folds'])

        unique_s = torch.unique(s_tr)
        remap = torch.zeros(int(unique_s.max()) + 1, dtype=torch.long, device=DEVICE)
        remap[unique_s] = torch.arange(unique_s.size(0), device=DEVICE)
        s_tr_remap = remap[s_tr]
        n_domains = int(unique_s.size(0))

        use_dann = (variant == 'dann')
        model = HardEasyModel(n_classes, use_dann=use_dann, n_domains=n_domains)

        t0 = time.time()
        model, best_bal, best_ep = train_fold_r5(
            model, X_tr, y_tr, s_tr_remap, X_te, y_te,
            variant, subj_weights,
            CONFIG['epochs'], CONFIG['lr'], CONFIG['weight_decay'],
            CONFIG['early_stop_patience'],
            CONFIG['dann_lambda_max'], CONFIG['dann_ramp_epochs'], CONFIG['mmd_lambda'],
            CONFIG['batch_size'], n_classes,
            CONFIG['confidence_threshold'], CONFIG['he_alpha'],
            CONFIG['reliability_warmup'])
        elapsed = time.time() - t0

        y_true, y_pred, _ = evaluate_dual(model, X_te, y_te,
                                          alpha_blend=CONFIG['he_alpha'],
                                          batch_size=CONFIG['batch_size']*2)
        fold_results[held_out] = {
            'sid': int(held_out),
            'y_true': y_true, 'y_pred': y_pred,
            'best_epoch': best_ep, 'elapsed_s': elapsed,
            'acc': accuracy_score(y_true, y_pred),
            'bal_acc': balanced_accuracy_score(y_true, y_pred),
            'reliability': subj_rel,
        }

        with open(ckpt_path, 'wb') as f:
            pickle.dump(fold_results, f)

        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return fold_results

## 8. Main grid — 3 tasks × 3 variants × 32 folds

In [10]:
ALL_RESULTS = {}
for task in CONFIG['tasks']:
    for variant in CONFIG['variants']:
        key = (task, variant)
        print(f'\n{"="*60}\n{task} | {variant}\n{"="*60}')
        ALL_RESULTS[key] = run_loso(task, variant)


valence | plain


valence_plain:   0%|          | 0/32 [00:00<?, ?it/s]

  computing reliability weights for held-out S1...
  computing reliability weights for held-out S2...
  computing reliability weights for held-out S3...
  computing reliability weights for held-out S4...
  computing reliability weights for held-out S5...
  computing reliability weights for held-out S6...
  computing reliability weights for held-out S7...
  computing reliability weights for held-out S8...
  computing reliability weights for held-out S9...
  computing reliability weights for held-out S10...
  computing reliability weights for held-out S11...
  computing reliability weights for held-out S12...
  computing reliability weights for held-out S13...
  computing reliability weights for held-out S14...
  computing reliability weights for held-out S15...
  computing reliability weights for held-out S16...
  computing reliability weights for held-out S17...
  computing reliability weights for held-out S18...
  computing reliability weights for held-out S19...
  computing reliabili

valence_dann:   0%|          | 0/32 [00:00<?, ?it/s]

  computing reliability weights for held-out S1...
  computing reliability weights for held-out S2...
  computing reliability weights for held-out S3...
  computing reliability weights for held-out S4...
  computing reliability weights for held-out S5...
  computing reliability weights for held-out S6...
  computing reliability weights for held-out S7...
  computing reliability weights for held-out S8...
  computing reliability weights for held-out S9...
  computing reliability weights for held-out S10...
  computing reliability weights for held-out S11...
  computing reliability weights for held-out S12...
  computing reliability weights for held-out S13...
  computing reliability weights for held-out S14...
  computing reliability weights for held-out S15...
  computing reliability weights for held-out S16...
  computing reliability weights for held-out S17...
  computing reliability weights for held-out S18...
  computing reliability weights for held-out S19...
  computing reliabili

...

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  computing reliability weights for held-out S31...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  computing reliability weights for held-out S32...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## 9. Aggregate metrics

In [13]:
def aggregate_fold(fold_results, task):
    n_classes = get_n_classes(task)
    per_fold, all_true, all_pred = [], [], []
    for sid, r in sorted(fold_results.items()):
        y_true, y_pred = r['y_true'], r['y_pred']
        all_true.append(y_true); all_pred.append(y_pred)
        per_fold.append({
            'sid': r['sid'],
            'acc': 100 * accuracy_score(y_true, y_pred),
            'bal': 100 * balanced_accuracy_score(y_true, y_pred),
            'f1_w': 100 * f1_score(y_true, y_pred, average='weighted', zero_division=0),
        })
    df = pd.DataFrame(per_fold)
    y_true_all = np.concatenate(all_true); y_pred_all = np.concatenate(all_pred)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=list(range(n_classes)))
    chance = 100.0 / n_classes
    t_stat, p_val = ttest_1samp(df['bal'], chance, alternative='greater')
    summary = {
        'bal_mean': df['bal'].mean(), 'bal_std': df['bal'].std(),
        'acc_mean': df['acc'].mean(), 'p_value': p_val,
        'above_chance': p_val < 0.05,
    }
    return df, cm, summary

rows, per_fold_all, cms_all = [], {}, {}
for (task, variant), fr in ALL_RESULTS.items():
    df, cm, summ = aggregate_fold(fr, task)
    per_fold_all[(task, variant)] = df
    cms_all[(task, variant)] = cm
    rows.append({'task': task, 'variant': variant, **summ})

SUMMARY = pd.DataFrame(rows)
SUMMARY.to_csv(RESULTS_DIR / 'summary_all_runs.csv', index=False)
print(SUMMARY.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


   task variant  bal_mean  bal_std  acc_mean      p_value  above_chance
valence   plain 55.859606 5.202986 55.275079 2.145943e-07          True
valence    dann 53.773797 3.984854 52.993041 3.854900e-06          True
valence     mmd 55.690297 5.645070 53.888524 1.435072e-06          True
arousal   plain 54.296331 3.926931 53.704701 3.587441e-07          True
arousal    dann 53.889822 3.972744 52.966780 2.290862e-06          True
arousal     mmd 51.362294 3.077761 50.112920 8.879474e-03          True
 3class   plain 38.076922 3.776291 38.255646 2.764645e-08          True
 3class    dann 37.012518 4.478077 41.130515 2.940679e-05          True
 3class     mmd 38.901027 6.217176 43.174238 8.890301e-06          True


## 10. Save per-fold results

In [14]:
for key, df in per_fold_all.items():
    df.to_csv(RESULTS_DIR / f'per_fold_{"_".join(key)}.csv', index=False)
for key, cm in cms_all.items():
    np.save(RESULTS_DIR / f'cm_{"_".join(key)}.npy', cm)
print('Saved to:', RESULTS_DIR)

Saved to: /content/drive/MyDrive/case_study_2/results/report5_perwindow
